<a href="https://colab.research.google.com/github/Moquiuti/Arquiteturas-RAG-com-LLMs-embeddings-busca-sem-ntica-e-cria-o-de-agentes-com-LangChain/blob/main/VectorStore.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q langchain langchain-community langchain-huggingface chromadb sentence-transformers

In [3]:
from langchain_core.documents import Document

documentos_brutos = [
    Document(
        page_content="""
        A TRATOTECH oferece garantia de 12 meses para notebooks, tablets e smartphones.
        A garantia cobre defeitos de fabricação, falhas internas e problemas técnicos não causados por mau uso.
        """,
        metadata={
            "origem": "manual_garantia.txt",
            "secao": "garantia",
            "categoria": "produto"
        }
    ),
    Document(
        page_content="""
        Produtos com queda, contato com líquido, violação do lacre ou sinais de mau uso
        não são cobertos pela garantia. Nesses casos, o suporte pode oferecer orçamento para reparo.
        """,
        metadata={
            "origem": "manual_garantia.txt",
            "secao": "exclusoes",
            "categoria": "produto"
        }
    ),
    Document(
        page_content="""
        O cliente pode solicitar cancelamento da compra em até 7 dias após o recebimento do produto,
        conforme a política de arrependimento. Após esse prazo, o pedido será avaliado pelo suporte.
        """,
        metadata={
            "origem": "politica_vendas.txt",
            "secao": "cancelamento",
            "categoria": "vendas"
        }
    ),
    Document(
        page_content="""
        O suporte técnico da TRATOTECH funciona de segunda a sexta-feira, das 8h às 18h.
        Atendimentos fora desse horário serão registrados e respondidos no próximo dia útil.
        """,
        metadata={
            "origem": "faq_suporte.txt",
            "secao": "atendimento",
            "categoria": "suporte"
        }
    ),
    Document(
        page_content="""
        Pagamentos com cartão de crédito podem ser parcelados em até 10 vezes.
        Pagamentos via Pix possuem confirmação mais rápida e podem liberar o envio do produto no mesmo dia.
        """,
        metadata={
            "origem": "politica_pagamento.txt",
            "secao": "pagamento",
            "categoria": "financeiro"
        }
    )
]

print(f"Total de documentos brutos: {len(documentos_brutos)}")

Total de documentos brutos: 5


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=["\n\n", "\n", ".", " ", ""]
)

chunks = text_splitter.split_documents(documentos_brutos)

for i, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = i
    chunk.metadata["estrategia_chunking"] = "recursive_character"
    chunk.metadata["chunk_size"] = 300
    chunk.metadata["chunk_overlap"] = 50

print(f"Total de chunks gerados: {len(chunks)}")

for chunk in chunks:
    print("\n--- CHUNK ---")
    print(chunk.page_content)
    print(chunk.metadata)

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Modelo de embeddings carregado com sucesso!")

In [6]:
from langchain_community.vectorstores import Chroma

persist_directory = "chroma_tratotech_db"

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=persist_directory
)

print("VectorStore criada com sucesso usando ChromaDB!")

VectorStore criada com sucesso usando ChromaDB!


In [7]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

print("Retriever configurado com sucesso!")

Retriever configurado com sucesso!


In [8]:
consulta = "Meu notebook caiu na água. A garantia cobre o conserto?"

resultados = retriever.invoke(consulta)

print(f"Consulta: {consulta}")

for i, doc in enumerate(resultados, start=1):
    print(f"\n--- Resultado {i} ---")
    print("Origem:", doc.metadata.get("origem"))
    print("Seção:", doc.metadata.get("secao"))
    print("Categoria:", doc.metadata.get("categoria"))
    print("Chunk ID:", doc.metadata.get("chunk_id"))
    print("Texto recuperado:")
    print(doc.page_content)

Consulta: Meu notebook caiu na água. A garantia cobre o conserto?

--- Resultado 1 ---
Origem: manual_garantia.txt
Seção: garantia
Categoria: produto
Chunk ID: 0
Texto recuperado:
A TRATOTECH oferece garantia de 12 meses para notebooks, tablets e smartphones.
        A garantia cobre defeitos de fabricação, falhas internas e problemas técnicos não causados por mau uso.

--- Resultado 2 ---
Origem: manual_garantia.txt
Seção: exclusoes
Categoria: produto
Chunk ID: 1
Texto recuperado:
Produtos com queda, contato com líquido, violação do lacre ou sinais de mau uso
        não são cobertos pela garantia. Nesses casos, o suporte pode oferecer orçamento para reparo.

--- Resultado 3 ---
Origem: politica_vendas.txt
Seção: cancelamento
Categoria: vendas
Chunk ID: 2
Texto recuperado:
O cliente pode solicitar cancelamento da compra em até 7 dias após o recebimento do produto,
        conforme a política de arrependimento. Após esse prazo, o pedido será avaliado pelo suporte.


In [9]:
consultas = [
    "Posso cancelar minha compra depois que recebi o produto?",
    "Qual é o horário de atendimento do suporte?",
    "Dá para parcelar no cartão?",
    "Meu celular está com defeito de fábrica. Tenho garantia?"
]

for consulta in consultas:
    print("\n" + "="*100)
    print("Consulta:", consulta)

    resultados = retriever.invoke(consulta)

    for i, doc in enumerate(resultados, start=1):
        print(f"\n--- Resultado {i} ---")
        print("Origem:", doc.metadata.get("origem"))
        print("Seção:", doc.metadata.get("secao"))
        print("Categoria:", doc.metadata.get("categoria"))
        print("Chunk ID:", doc.metadata.get("chunk_id"))
        print(doc.page_content)


Consulta: Posso cancelar minha compra depois que recebi o produto?

--- Resultado 1 ---
Origem: politica_vendas.txt
Seção: cancelamento
Categoria: vendas
Chunk ID: 2
O cliente pode solicitar cancelamento da compra em até 7 dias após o recebimento do produto,
        conforme a política de arrependimento. Após esse prazo, o pedido será avaliado pelo suporte.

--- Resultado 2 ---
Origem: manual_garantia.txt
Seção: exclusoes
Categoria: produto
Chunk ID: 1
Produtos com queda, contato com líquido, violação do lacre ou sinais de mau uso
        não são cobertos pela garantia. Nesses casos, o suporte pode oferecer orçamento para reparo.

--- Resultado 3 ---
Origem: politica_pagamento.txt
Seção: pagamento
Categoria: financeiro
Chunk ID: 4
Pagamentos com cartão de crédito podem ser parcelados em até 10 vezes.
        Pagamentos via Pix possuem confirmação mais rápida e podem liberar o envio do produto no mesmo dia.

Consulta: Qual é o horário de atendimento do suporte?

--- Resultado 1 ---
Ori

In [10]:
consulta = "O produto molhou. A garantia ainda vale?"

resultados_com_score = vectorstore.similarity_search_with_score(
    consulta,
    k=3
)

print(f"Consulta: {consulta}")

for i, (doc, score) in enumerate(resultados_com_score, start=1):
    print(f"\n--- Resultado {i} ---")
    print("Score:", score)
    print("Origem:", doc.metadata.get("origem"))
    print("Seção:", doc.metadata.get("secao"))
    print("Categoria:", doc.metadata.get("categoria"))
    print("Texto:")
    print(doc.page_content)

Consulta: O produto molhou. A garantia ainda vale?

--- Resultado 1 ---
Score: 0.8533916473388672
Origem: manual_garantia.txt
Seção: exclusoes
Categoria: produto
Texto:
Produtos com queda, contato com líquido, violação do lacre ou sinais de mau uso
        não são cobertos pela garantia. Nesses casos, o suporte pode oferecer orçamento para reparo.

--- Resultado 2 ---
Score: 0.9879553318023682
Origem: politica_vendas.txt
Seção: cancelamento
Categoria: vendas
Texto:
O cliente pode solicitar cancelamento da compra em até 7 dias após o recebimento do produto,
        conforme a política de arrependimento. Após esse prazo, o pedido será avaliado pelo suporte.

--- Resultado 3 ---
Score: 1.2104551792144775
Origem: manual_garantia.txt
Seção: garantia
Categoria: produto
Texto:
A TRATOTECH oferece garantia de 12 meses para notebooks, tablets e smartphones.
        A garantia cobre defeitos de fabricação, falhas internas e problemas técnicos não causados por mau uso.
